<a href="https://colab.research.google.com/github/nashranoor98/hospital-readmission-prediction/blob/main/CaseStudy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Importing and Studying Data

In [ ]:
import pandas as pd
import numpy as np

# Load the UCI Diabetes 130-US Hospitals dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00296/dataset_diabetes.zip"
data = pd.read_csv("data/diabetic_data.csv")
data = data.replace("?", np.nan)

In [ ]:
print(data.shape)
print(data.head())

In [ ]:
print(data.info())
print(data.describe(include="all").T.head(15))

In [ ]:
print(data.isnull().sum().sort_values(ascending=False).head(15))
print("Duplicate rows:", data.duplicated().sum())

2. EDA and Visualisation

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(x=data["readmitted"])
plt.title("Readmission Distribution")
plt.xlabel("Readmission Category")
plt.ylabel("Number of Patients")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(data["time_in_hospital"], bins=14, kde=True)
plt.title("Distribution of Time in Hospital")
plt.xlabel("Days in Hospital")
plt.show()

3. Preparing Dataset

In [ ]:
data["target_30day"] = (data["readmitted"] == "<30").astype(int)

drop_cols = ["encounter_id", "patient_nbr", "readmitted", "target_30day"]
X = data.drop(columns=drop_cols)
y = data["target_30day"]

# Remove columns with more than 90% missing values
high_missing = X.columns[X.isna().mean() > 0.90]
X = X.drop(columns=high_missing)

print("Target counts:")
print(y.value_counts())

4. Splitting Dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

numeric = X_train.select_dtypes(include=np.number).columns
categorical = X_train.select_dtypes(exclude=np.number).columns

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical)
])

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

5. Training Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(penalty="l2", C=1.0, max_iter=1000, solver="liblinear"))
])

model.fit(X_train, y_train)
print("Model training completed.")

6. Prediction and Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", round(auc, 4))
print(classification_report(y_test, y_pred, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve - 30-Day Readmission")
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix")
plt.show()

7. False Negative vs False Positive Cost

A false negative means a patient who is readmitted within 30 days is not flagged by the model. A false positive means a patient is flagged even though they are not readmitted.

In a hospital setting, false negatives may represent missed opportunities for additional follow-up, while false positives can increase workload for care teams. The appropriate classification threshold therefore depends on the relative operational cost of these errors.

This notebook is an educational case study and is not a clinical decision-support system.